# 🚀 DataMate — Public Test Inference trên GPU Local (RTX 5060 8GB)
Notebook này được thiết kế để chạy suy luận trực tiếp trên GPU máy tính của bạn sử dụng Virtual Environment:
`E:\ZALO_AI_Challenge\zalo-vecna\venv`

- **GPU:** NVIDIA GeForce RTX 5060 Laptop GPU (8GB VRAM)
- **Chế độ:** 4-bit Quantization (`--load_in_4bit`) tối ưu VRAM ~4.5GB
- **Cơ chế:** Tự động **chạy nối tiếp từ câu 67** (giữ nguyên 66 câu đã chạy từ trước)

In [ ]:
# 1. Kiểm tra môi trường GPU & PyTorch CUDA
import sys
import torch

print(f"Python executable: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM: {total_vram:.2f} GB")
else:
    print("[!] Cảnh báo: PyTorch chưa nhận CUDA GPU!")

## 2. Kiểm tra tiến độ file submission và checkpoints có sẵn

In [ ]:
# 2. Kiểm tra các câu đã hoàn thành trước đó
import os
import json

sub_path = "submission_grpo.jsonl"
if os.path.exists(sub_path):
    with open(sub_path, "r", encoding="utf-8") as f:
        done = [json.loads(line)["id"] for line in f if line.strip()]
    print(f"[*] Đã có sẵn {len(done)} câu trong {sub_path}.")
    print(f"[*] Script sẽ tự động chạy nối tiếp từ câu {len(done)+1} đến 1,000!")
else:
    print(f"[*] Chưa có {sub_path}, sẽ chạy từ câu 1 đến 1,000.")

for ad in ["checkpoints/grpo_adapter", "checkpoints/sft_adapter"]:
    if os.path.exists(ad):
        print(f"[+] Tìm thấy adapter: {ad}")
    else:
        print(f"[!] Chưa tìm thấy: {ad}")

## 3. Chạy suy luận Public Test trên GPU RTX 5060
Sử dụng `--load_in_4bit` và `--batch_size 2` để đảm bảo bộ nhớ VRAM luôn dưới 5GB và chạy ổn định.

In [ ]:
# 3. Chạy suy luận nối tiếp với GRPO adapter trên GPU Local
import sys
!{sys.executable} run_public_test.py \
    --data_dir public_test \
    --output submission_grpo.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/grpo_adapter \
    --batch_size 2 \
    --load_in_4bit

## 4. Kiểm tra format & xuất file `submission.jsonl` chuẩn để nộp bài

In [ ]:
# 4. Thẩm định và nhân bản thành file submission.jsonl đúng chuẩn
import os
import json
import shutil
from collections import Counter

sub_path = "submission_grpo.jsonl"
target_path = "submission.jsonl"

if os.path.exists(sub_path):
    with open(sub_path, "r", encoding="utf-8") as f:
        lines = [json.loads(line) for line in f if line.strip()]
    print(f"[+] Tổng số câu trong {sub_path}: {len(lines)} / 1000")
    print("Phân bố các kiểu câu trả lời:")
    for k, v in Counter(x.get("type") for x in lines).items():
        print(f"  {k:15s}: {v}")
    
    # Nhân bản thành submission.jsonl
    shutil.copyfile(sub_path, target_path)
    print(f"\n[+] ĐÃ TẠO FILE NỘP BÀI: {target_path} ({os.path.getsize(target_path)} bytes)!")
    print("Bạn có thể upload file 'submission.jsonl' này lên cổng thi Zalo AI!")
else:
    print(f"[!] Không tìm thấy file {sub_path}.")